In [35]:
import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [36]:
engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [53]:

query = f"""
	SELECT nombre_cliente AS NOMBRES,agencia_atencion AS Agencia_comercial,dni_cliente as NUMERO_DOCUMENTO
    ,celular AS cl_telf1
    FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-09-04 10:00:00'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)

ruta_archivo = os.path.join(ruta_alfin, 'evaluar_alfin.csv')
df_correo_ref.to_csv(ruta_archivo, sep=';')

In [54]:

filename='evaluar_alfin.csv'
df_boqueo=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

In [55]:


filename='Consulta_de_Campañas_202608_V3_SS_02.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_01.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_03.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_04.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_03=df_validar_03.drop('TASA_MIN_DESCUENTO')
df_validar_04=df_validar_04.drop('TASA_MIN_DESCUENTO')

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)

df_validar=df_validar.dropDuplicates(['DNI'])
df_validar=df_validar.withColumnRenamed('DNI','NUMERO_DOCUMENTO')

In [56]:
df_boqueo=df_boqueo.join(df_validar,['NUMERO_DOCUMENTO'],'inner')
df_boqueo.count()

5106

In [41]:
df_boqueo.show(2)

+----------------+---+--------------------+--------------------+---------+------------+-----------+--------+------------+-------------------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+----------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+-----------+
|NUMERO_DOCUMENTO|_c0|             NOMBRES|   Agencia_comercial| cl_telf1| COLOR_FINAL|COD_USER_V3| USER_V3|   PERFIL_RO|            campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|T

In [7]:
df_boqueo_pd=df_boqueo.toPandas()

In [18]:
ruta_archivo = os.path.join(ruta_alfin, 'evaluar_alfin_1.csv')
df_boqueo_pd.to_csv(ruta_archivo, sep=';', index=False)

In [19]:

filename='evaluar_alfin_1.csv'
df_boqueo_1=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)


In [20]:


engine_mysql_1 = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

In [93]:
query = """
SELECT *
FROM crm_target.alfin_clientes
WHERE cl_base = 'setiembre 2026'
limit 1
"""

chunks = pd.read_sql(
    query,
    engine_mysql_1,
    chunksize=10000
)

df_valentina = pd.concat(chunks, ignore_index=True)

In [27]:
df_boqueo_1_pd=df_boqueo_1.toPandas()

{'TASA_2', 'OFERTA_SS', 'TASA_4_SS', 'TASA_1', 'numentidades', '_c0', 'FLAG_DEUDA_V_OFERTA', 'TASA_5_SS', 'COLOR_FINAL', 'MARCA_PD', 'TASA_6_SS', 'TASA_3_SS', 'TASA_7_SS', 'TASA_5', 'COD_USER_V3', 'TASA_7', 'campaña', 'TIPO', 'TOTAL_A_LIQUIDAR', 'PERFIL_ESPECIAL', 'TASA_1_SS', 'AUTORIZACION_DATOS', 'TASA_3', 'PROPENSION_DISTRIBUCION', 'TASA_4', 'TASA_CREDITO_ANTERIOR', 'ALERTA_MAQUETA', 'rango_deuda', 'TASA_6', 'FEN', 'TASA_2_SS'}


In [87]:
columnas = [
    'TASA_2','TASA_1', 'numentidades', 'FLAG_DEUDA_V_OFERTA', 'COLOR_FINAL', 'TASA_5', 'TASA_7', 'campaña', 'TASA_3', 'PROPENSION_DISTRIBUCION', 'TASA_4','TASA_6','NOMBRES','Agencia_comercial','NUMERO_DOCUMENTO','cl_telf1','FRESCURA'

]

df_boqueo_2_pd = (
    df_boqueo_1_pd.loc[:, columnas]
    .rename(columns={
        "DNI": "NUMERO_DOCUMENTO",
        "nombres": "NOMBRES",
        "celular": "cl_telf1",
        "COLOR_FINAL": "color_final",
        "TASA_1": "Tasa_1",
        "TASA_2": "Tasa_2",
        "TASA_3": "Tasa_3",
        "TASA_4": "Tasa_4",
        "TASA_5": "Tasa_5",
        "TASA_6": "Tasa_6",
        "TASA_7": "Tasa_7",
        "campaña": "campania",
        "PROPENSION_DISTRIBUCION": "PROPENSION_IC",
        "agencia": "Agencia_comercial"
    })
)

In [88]:
df_boqueo_2_pd['cl_estado']='1'
df_boqueo_2_pd['estado']='ACTIVO'
df_boqueo_2_pd['cl_base']='setiembre 2026'

In [89]:
# bloqueo=set(df_boqueo_1.columns)
bloqueo=set(df_boqueo_2_pd.columns.tolist())
vale=set(df_valentina.columns.tolist())

print(bloqueo&vale)


{'NOMBRES', 'Tasa_1', 'cl_telf1', 'estado', 'cl_base', 'Tasa_3', 'cl_estado', 'PROPENSION_IC', 'Agencia_comercial', 'Tasa_5', 'Tasa_4', 'Tasa_2', 'FRESCURA', 'NUMERO_DOCUMENTO', 'color_final', 'campania', 'Tasa_7', 'Tasa_6'}


In [90]:

df_boqueo_2_pd.to_sql(
    name="alfin_clientes",
    con=engine_mysql_1,
    if_exists="append",
    index=False,
    chunksize=1000
)

5106

In [96]:
import pandas as pd

lista_df = []

limit = 5000
offset = 0

while True:

    query = f"""
    SELECT *
    FROM crm_target.alfin_clientes
    WHERE cl_base = 'setiembre 2026'
    LIMIT {limit} OFFSET {offset}
    """

    df_temp = pd.read_sql(query, engine_mysql_1)

    if df_temp.empty:
        break

    lista_df.append(df_temp)

    print(f"Descargados: {offset + len(df_temp)}")

    offset += limit

df_valentina = pd.concat(lista_df, ignore_index=True)

Descargados: 5000
Descargados: 10000
Descargados: 15000
Descargados: 20000
Descargados: 25000
Descargados: 30000
Descargados: 35000
Descargados: 40000
Descargados: 45000
Descargados: 50000
Descargados: 55000
Descargados: 60000
Descargados: 65000
Descargados: 65514


C:\Users\Data\AppData\Local\Temp\ipykernel_9320\1930674594.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_valentina = pd.concat(lista_df, ignore_index=True)


In [97]:
df_valentina=df_valentina.merge(
df_boqueo_2_pd[['NUMERO_DOCUMENTO']],on='NUMERO_DOCUMENTO',how='inner'
)
df_valentina.shape

(8595, 178)

In [98]:
df_valentina['uno']='12'

df_nuevo = df_valentina[
    ["cl_telf1", "cl_id", "uno", "NOMBRES"]
].copy()
df_nuevo["todo"] = (
    df_nuevo[["cl_telf1", "cl_id", "uno", "NOMBRES"]]
    .fillna("")
    .astype(str)
    .agg(",".join, axis=1)
)

df_nuevo = df_nuevo[["todo"]]

In [ ]:
df_nuevo.head()

,todo
0,"989478087,3525316,12,CARMEN BERTHA HINOSTROZA ..."
1,"969247562,3525322,12,BLANCA GLADIS PELAEZ CAST..."
2,"941189846,3525335,12,LUCILA YANNETTE CASTILLO ..."
3,"961939100,3525341,12,ANTONIO PEDRO FLORES MUÑO..."
4,"992259295,3525344,12,SANDRA VERONICA ZAMORA TE..."


In [101]:
ruta_archivo = os.path.join(ruta_alfin, 'subir.csv')
df_nuevo.to_csv(ruta_archivo, sep=';')

In [106]:
df_valentina[df_valentina['cl_id']=='3702718']

,cl_id,TIPO_DOI,NUMERO_DOCUMENTO,NOMBRES,APELLIDO_PATERNO,APELLIDO_MATERNO,SUCURSAL,TIENDA,DEPARTAMENTO,PROVINCIA,...,p_banco,tasa_minima,MES_DURACION_BASE,ANIO_DURACION_BASE,PERFIL_GLOBAL,FLG_AAHH,SCORE_TELEFONO,BLOQUE,INTENSIDAD_MAX,uno


In [105]:
print(df_valentina.columns.tolist())

['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', '

In [ ]:
['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'cl_estado', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'tasa_minima', 'MES_DURACION_BASE', 'ANIO_DURACION_BASE', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'BLOQUE', 'INTENSIDAD_MAX', 'uno']tel

### subir a v

In [43]:
query = f"""
    select * from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
    where Fecha_Envio>='2026-09-01'

"""
df_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [ ]:
['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'SERVICIO', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'fecha', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FLAT2', 'REP1', 'REP2', 'PILOTO_RETENCION', 'CAMP_BONO', 'ACCION']
z

['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUEL

In [67]:
df_boqueo=df_boqueo.withColumn('CRUCE',F.lit('CET'))
df_boqueo=df_boqueo.withColumn('lote',F.lit('INVENTARIO'))
df_boqueo=df_boqueo.withColumn('cl_base',F.lit('INVENTARIO'))
df_boqueo=df_boqueo.withColumn("estado",F.lit('ACTIVO'))
df_boqueo=df_boqueo.withColumn("Fecha_Envio",F.lit('2026-09-01'))

In [58]:
df_boqueo=df_boqueo.withColumnRenamed('campaña','campania')
df_boqueo=df_boqueo.withColumnRenamed('TASA_1','Tasa_1')
df_boqueo=df_boqueo.withColumnRenamed('TASA_2','Tasa_2')
df_boqueo=df_boqueo.withColumnRenamed('TASA_3','Tasa_3')
df_boqueo=df_boqueo.withColumnRenamed('TASA_4','Tasa_4')
df_boqueo=df_boqueo.withColumnRenamed('TASA_5','Tasa_5')
df_boqueo=df_boqueo.withColumnRenamed('TASA_6','Tasa_6')
df_boqueo=df_boqueo.withColumnRenamed('TASA_7','Tasa_7')
df_boqueo=df_boqueo.withColumnRenamed('COLOR_FINAL','color_final')
df_boqueo=df_boqueo.withColumnRenamed('PROPENSION_DISTRIBUCION','PROPENSION_IC')


In [62]:

df_base_set=set(df_base.columns)
bloque_Set=set(df_boqueo.columns)
print(bloque_Set-df_base_set)
print(bloque_Set & df_base_set)
print(df_base_set)


{'cl_estado'}
{'CAPACIDAD_MAX', 'cl_base', 'Tasa_5', 'NOMBRES', 'NUMERO_DOCUMENTO', 'color_final', 'Tasa_7', 'PROPENSION_IC', 'FRESCURA', 'GRUPO_TASA', 'Tasa_3', 'campania', 'Tasa_6', 'MGNEG', 'TIPO_BASE', 'USER_V3', 'PLAZO', 'Tasa_4', 'CRUCE', 'OFERTA_MAX', 'PERFIL_RO', 'Tasa_1', 'cl_telf1', 'lote', 'Agencia_comercial', 'Tasa_2'}
{'DEPARTAMENTO', 'Tasa_5', 'Ubicacion', 'TEM', 'cl_movil', 'color_final', 'flag_deuda_v_oferta', 'cl_telf7', 'cl_fecha_gestion', 'tipo_cliente_riegos', 'nombre_base', 'Entidad_3', 'MONTO_DESEMBOLSADO', 'cl_gestor', 'Oferta_12M', 'Tasa_6', 'RANGO_OFERTA', 'LOCALIDAD', 'SBI', 'TIPO_CLIENTE', 'PREST_PREVIO', 'REP2', 'cl_accion_ant', 'OFERTA_REEN', 'Desgravamen_24M', 'sucursal_comercial', 'RETIRO_DESEMBOLSO', 'CUOTA', 'MEJOR_TIPIFICACION', 'cl_celular', 'PROMOCION', 'Deuda_1', 'SUCURSAL', 'CUOTA_24M', 'cl_hora_gestion', 'MES_GESTION', 'cl_orden', 'RANGO_EDAD2', 'cl_telf1', 'CLIENTE_NUEVO', 'MES_DURACION_BASE', 'Oferta_18M', 'PROMOCION2', 'lote', 'cl_telf5', 'INTE

In [69]:
df_boqueo=df_boqueo.drop('numentidades', 'FLAG_DEUDA_V_OFERTA', 'ALERTA_MAQUETA', 'TASA_CREDITO_ANTERIOR', 'AUTORIZACION_DATOS', 'rango_deuda', 'TASA_5_SS', 'TASA_2_SS', 'TASA_7_SS', 'TASA_4_SS', 'TOTAL_A_LIQUIDAR', 'TIPO', 'COD_USER_V3', 'MARCA_PD', 'PERFIL_ESPECIAL', 'TASA_6_SS', 'TASA_3_SS', 'FEN', '_c0', 'estado', 'OFERTA_SS', 'TASA_1_SS'
,'cl_estado','estado')


In [16]:
print(df_base_set)

{'DEPARTAMENTO', 'Tasa_5', 'Ubicacion', 'TEM', 'cl_movil', 'color_final', 'flag_deuda_v_oferta', 'cl_telf7', 'cl_fecha_gestion', 'tipo_cliente_riegos', 'nombre_base', 'Entidad_3', 'MONTO_DESEMBOLSADO', 'cl_gestor', 'Oferta_12M', 'Tasa_6', 'RANGO_OFERTA', 'LOCALIDAD', 'SBI', 'TIPO_CLIENTE', 'PREST_PREVIO', 'REP2', 'cl_accion_ant', 'OFERTA_REEN', 'Desgravamen_24M', 'sucursal_comercial', 'RETIRO_DESEMBOLSO', 'CUOTA', 'MEJOR_TIPIFICACION', 'cl_celular', 'PROMOCION', 'Deuda_1', 'SUCURSAL', 'CUOTA_24M', 'cl_hora_gestion', 'MES_GESTION', 'cl_orden', 'RANGO_EDAD2', 'cl_telf1', 'CLIENTE_NUEVO', 'MES_DURACION_BASE', 'Oferta_18M', 'PROMOCION2', 'lote', 'cl_telf5', 'INTENSIDAD_MAX', 'color', 'cl_telf8', 'ID_CLIENTE', 'Desgravamen_12M', 'incremento_monto_riesgos', 'Tasa_2', 'PEER', 'RANGO_OFERTA2', 'FECHA_SOL', 'NUEVOS_4M', 'cl_telf9', 'cl_mes', 'Nombre_prioridad', 'BASE', 'NUMERO_DOCUMENTO', 'FLAT2', 'FEC_NACIMIENTO', 'cl_asesor', 'Deuda_3', 'Tasa_3', 'Desgravamen_18M', 'campania', 'cl_accion', 'N

In [70]:
append_table_SQL(spark,df_boqueo,f'Base_Maestra_Alfin_bk',server_kishin,user_kishin,pwd_kishin,'DANTALION')
